# HI 5312 – Introduction to Data Analysis
## Module 3: Retrieving Healthcare Data with SQL and PostgreSQL

**Estimated time:** 6–8 hours  
**Environment:** GitHub Codespaces  
**Database:** PostgreSQL (`hi5304`)  
**Primary tools:** Terminal, `psql`, SQL, Python, pandas, and Jupyter Notebook

> **Course progression:** Better Questions → Better Data → Better Analysis → Better Decisions → Better Health

## Clinical Request

The Director of Quality Improvement has decided that the spreadsheet used in Module 2 is not sufficient for the readmissions investigation. The organization stores patient, medication, and blood-pressure data in a PostgreSQL database.

Your new request is:

> **Retrieve the patients and clinical observations needed to investigate high blood pressure and related risk factors.**

To complete the request, you must first understand the database, inspect its tables, write SQL queries in the terminal, and then execute those same queries from a Jupyter Notebook.

## Why This Matters

Healthcare data are often stored in relational databases rather than individual spreadsheets. SQL allows analysts to retrieve only the records and variables needed for a specific question, combine information from multiple tables, summarize large datasets, and create reproducible cohorts.

A health informatics analyst does not need to become a database administrator, but must be able to communicate clearly with data engineers and write accurate, transparent queries.

## Learning Objectives

After completing this module, you should be able to:

1. Explain the purpose of relational databases in healthcare.
2. Identify databases, schemas, tables, rows, columns, primary keys, and foreign keys.
3. Connect to PostgreSQL from the Codespaces terminal.
4. Inspect available tables and table structures with `psql`.
5. Retrieve columns with `SELECT` and `FROM`.
6. Filter records with `WHERE`, comparison operators, and logical operators.
7. Sort and limit results with `ORDER BY` and `LIMIT`.
8. Summarize data with aggregate functions and `GROUP BY`.
9. Filter grouped results with `HAVING`.
10. Combine related tables with `INNER JOIN` and `LEFT JOIN`.
11. Connect to PostgreSQL from Python.
12. Load SQL query results into pandas DataFrames.
13. Document assumptions used to define a clinical cohort.

## Essential Question

> **How can I retrieve the correct healthcare data for the question I have been asked?**

## Professional Competency

**Construct and document SQL queries that retrieve clinically meaningful data from a relational healthcare database.**

# Part 1 – Relational Database Foundations

A relational database organizes information into related tables.

| Concept | Meaning |
|---|---|
| Database | An organized collection of related data |
| Schema | A named organizational layer within a database |
| Table | Data arranged in rows and columns |
| Row | One record or observation |
| Column | One variable or field |
| Primary key | A value that uniquely identifies a row |
| Foreign key | A field that links one table to another |

For example, `patient_id` may uniquely identify a patient in a `patients` table and also appear in `medications` and `bp_readings` as a foreign key.

## Data Analysis Toolbox: Think Before You Query

Before writing SQL, identify:

- the clinical or operational question;
- the unit of analysis;
- the population to include;
- the time period;
- the tables required;
- the variables required;
- the key used to join tables;
- inclusion and exclusion rules;
- how missing values will be handled.

A syntactically correct query can still answer the wrong question.

# Part 2 – PostgreSQL from the Terminal

The course database is already running inside the Codespace.

| Setting | Value |
|---|---|
| Host | `db` |
| Port | `5432` |
| Database | `hi5304` |
| Username | `hi5304_user` |
| Password | `hi5304` |

These credentials are intentionally simple for coursework. Real systems should use environment variables or secure secrets.

## Open a PostgreSQL Session

Open the Codespaces terminal and enter:

```bash
psql -h db -U hi5304_user -d hi5304
```

When prompted, enter:

```text
hi5304
```

Useful `psql` commands:

| Command | Purpose |
|---|---|
| `\l` | List databases |
| `\dn` | List schemas |
| `\dt` | List tables |
| `\d table_name` | Describe a table |
| `\x` | Toggle expanded display |
| `\q` | Exit `psql` |

`psql` commands beginning with a backslash are terminal commands, not SQL statements.

## Guided Laboratory 1 – Explore the Database in the Terminal

Complete these steps in the terminal:

1. Connect to the `hi5304` database.
2. List the schemas with `\dn`.
3. List the tables with `\dt`.
4. Describe the `patients` table with `\d patients`.
5. Describe `medications` and `bp_readings` if they are present.
6. Display five rows from `patients`:

```sql
SELECT *
FROM patients
LIMIT 5;
```

7. Exit with `\q`.

Record the primary key of each table and any fields that appear to link tables.

# Part 3 – Basic SQL

A basic SQL query uses `SELECT` to identify columns and `FROM` to identify the table.

```sql
SELECT patient_id, age, sex
FROM patients;
```

SQL keywords are conventionally written in uppercase. Table and column names are commonly lowercase.

## Selecting Records

Use an asterisk to return every column:

```sql
SELECT *
FROM patients;
```

For most analytical work, select only the columns you need:

```sql
SELECT patient_id, age, sex
FROM patients;
```

Use `LIMIT` while exploring large tables:

```sql
SELECT *
FROM patients
LIMIT 10;
```

## Aliases and Distinct Values

Aliases make output easier to read:

```sql
SELECT
    patient_id AS patient,
    age AS age_years
FROM patients;
```

Use `DISTINCT` to inspect unique values:

```sql
SELECT DISTINCT sex
FROM patients;
```

This is especially useful for detecting inconsistent categories.

## Guided Laboratory 2 – First Queries

Run SQL statements in `psql` to:

1. Display the first 10 rows of `patients`.
2. Retrieve only `patient_id`, `age`, and `sex`.
3. List distinct values for `sex`.
4. List distinct values for any smoking or diabetes variable available in the table.
5. Rename one output column with an alias.

For each query, write one sentence describing what one row represents.

# Part 4 – Filtering with WHERE

Use `WHERE` to restrict records.

```sql
SELECT patient_id, age, sex
FROM patients
WHERE age >= 65;
```

Common comparison operators:

| Operator | Meaning |
|---|---|
| `=` | Equal to |
| `<>` or `!=` | Not equal to |
| `>` | Greater than |
| `<` | Less than |
| `>=` | Greater than or equal to |
| `<=` | Less than or equal to |

## Logical and Special Operators

```sql
-- Two conditions must both be true
WHERE age >= 65 AND sex = 'F'

-- Either condition may be true
WHERE age < 40 OR age >= 75

-- Match one of several values
WHERE sex IN ('F', 'M')

-- Match a range, including both endpoints
WHERE age BETWEEN 50 AND 70

-- Pattern matching
WHERE medication_name LIKE '%statin%'

-- Missing data
WHERE bmi IS NULL
```

Never test missing values with `= NULL`. Use `IS NULL` or `IS NOT NULL`.

## Guided Laboratory 3 – Build a Cohort

Write queries to identify:

1. Patients age 65 years or older.
2. Patients between ages 45 and 64.
3. Patients with missing BMI, if BMI is available.
4. Patients who meet two risk-factor conditions.
5. Patients who do not meet a selected condition.

For each query, state the inclusion criteria in plain language before writing SQL.

# Part 5 – Sorting and Limiting Results

Use `ORDER BY` to sort records.

```sql
SELECT patient_id, age
FROM patients
ORDER BY age DESC;
```

Sort by more than one column:

```sql
SELECT patient_id, sex, age
FROM patients
ORDER BY sex ASC, age DESC;
```

Find the five oldest patients:

```sql
SELECT patient_id, age
FROM patients
ORDER BY age DESC
LIMIT 5;
```

# Part 6 – Aggregate Functions

Aggregate functions summarize many rows into one result.

| Function | Purpose |
|---|---|
| `COUNT()` | Count rows or nonmissing values |
| `AVG()` | Calculate the mean |
| `SUM()` | Calculate a total |
| `MIN()` | Find the minimum |
| `MAX()` | Find the maximum |

```sql
SELECT
    COUNT(*) AS patient_count,
    AVG(age) AS mean_age,
    MIN(age) AS minimum_age,
    MAX(age) AS maximum_age
FROM patients;
```

## GROUP BY and HAVING

Use `GROUP BY` to calculate a result for each category.

```sql
SELECT
    sex,
    COUNT(*) AS patient_count,
    AVG(age) AS mean_age
FROM patients
GROUP BY sex
ORDER BY patient_count DESC;
```

Use `HAVING` to filter groups after aggregation:

```sql
SELECT
    sex,
    COUNT(*) AS patient_count
FROM patients
GROUP BY sex
HAVING COUNT(*) >= 10;
```

`WHERE` filters rows before grouping. `HAVING` filters groups after grouping.

## Guided Laboratory 4 – Summarize the Population

Write queries to calculate:

1. Total number of patients.
2. Minimum, maximum, and average age.
3. Number of patients in each sex category.
4. Average age within each sex category.
5. Counts for a diabetes, smoking, or hypertension variable if present.
6. Categories containing at least 10 patients using `HAVING`.

Interpret each result in one sentence.

# Part 7 – Combining Tables with JOIN

Healthcare questions often require more than one table.

An `INNER JOIN` returns only records that match in both tables:

```sql
SELECT
    p.patient_id,
    p.age,
    b.reading_date,
    b.systolic,
    b.diastolic
FROM patients AS p
INNER JOIN bp_readings AS b
    ON p.patient_id = b.patient_id;
```

A `LEFT JOIN` keeps every row from the table on the left, even when no matching record exists on the right:

```sql
SELECT
    p.patient_id,
    p.age,
    m.medication_name
FROM patients AS p
LEFT JOIN medications AS m
    ON p.patient_id = m.patient_id;
```

## Join Safety Checklist

Before joining tables, confirm:

- the linking fields represent the same concept;
- the data types match;
- the relationship is one-to-one, one-to-many, or many-to-many;
- duplicate rows are expected or unintended;
- the selected join type matches the question;
- row counts are checked before and after the join.

A patient with several medications or blood-pressure readings will appear on several rows after a join. This may be correct, but it changes the unit of analysis.

## Guided Laboratory 5 – Join Clinical Tables

Using the tables available in the database:

1. Join `patients` to `bp_readings` using `patient_id`.
2. Return patient characteristics and blood-pressure measurements.
3. Sort measurements from highest to lowest systolic pressure.
4. Join `patients` to `medications`.
5. Compare the row count produced by an `INNER JOIN` with a `LEFT JOIN`.
6. Explain what one row represents after each join.

# My Professional Perspective

SQL is one of the most practical skills in health data science because it connects the analytical question directly to the underlying data. It allows analysts to define cohorts transparently, inspect data quality at the source, and retrieve only the information needed for a specific analysis. Understanding SQL also improves communication with database administrators, software engineers, clinicians, and research teams. The goal is not merely to produce a query that runs. The goal is to produce a query whose logic can be understood, reviewed, and reproduced.

# Part 8 – PostgreSQL in a Jupyter Notebook

The remainder of the module moves from the terminal to Python. SQL remains responsible for retrieving data, while pandas displays and analyzes the results.

The connection below uses the instructional Codespaces credentials. In production, credentials should be stored in environment variables or a secrets manager.

In [ ]:
# Install packages only if they are missing in your Codespace.
# Uncomment the next line if needed.
# %pip install pandas sqlalchemy psycopg2-binary

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

DB_HOST = os.getenv("DB_HOST", "db")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "hi5304")
DB_USER = os.getenv("DB_USER", "hi5304_user")
DB_PASSWORD = os.getenv("DB_PASSWORD", "hi5304")

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)
print(f"Prepared connection to {DB_NAME} on host {DB_HOST}.")

Prepared connection to hi5304 on host db.


In [2]:
# Test the connection.
with engine.connect() as connection:
    result = connection.execute(text("SELECT current_database(), current_user;"))
    print(result.fetchone())

('hi5304', 'hi5304_user')


## Inspect the Database from Python

The following query uses PostgreSQL's `information_schema` to list available tables.

In [3]:
tables_query = """
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
ORDER BY table_schema, table_name;
"""

tables = pd.read_sql(tables_query, engine)
tables

,table_schema,table_name
0,hi5304_training,admissions
1,hi5304_training,labs
2,hi5304_training,patients
3,mimiciv_hosp,admissions
4,mimiciv_hosp,d_hcpcs
5,mimiciv_hosp,d_icd_diagnoses
6,mimiciv_hosp,d_icd_procedures
7,mimiciv_hosp,d_labitems
8,mimiciv_hosp,diagnoses_icd
9,mimiciv_hosp,drgcodes


In [4]:
columns_query = """
SELECT
    table_name,
    ordinal_position,
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'public'
ORDER BY table_name, ordinal_position;
"""

columns = pd.read_sql(columns_query, engine)
columns

,table_name,ordinal_position,column_name,data_type
0,patients,1,patient_id,integer
1,patients,2,first_name,text
2,patients,3,last_name,text
3,patients,4,date_of_birth,date
4,patients,5,sex,text
5,patients,6,race,text
6,patients,7,ethnicity,text
7,patients,8,zip_code,text


## Run SQL and Return a pandas DataFrame

`pandas.read_sql()` executes the SQL statement and stores the result in a DataFrame.

In [5]:
patients_query = """
SELECT *
FROM patients
LIMIT 10;
"""

patients_preview = pd.read_sql(patients_query, engine)
patients_preview

,patient_id,first_name,last_name,date_of_birth,sex,race,ethnicity,zip_code
0,1001,John,Doe,1968-04-12,M,White,Non-Hispanic,75201
1,1002,Maria,Lopez,1975-09-30,F,Hispanic,Hispanic,75204
2,1003,James,Smith,1982-01-18,M,Black,Non-Hispanic,75080
3,1004,Linda,Chen,1990-06-05,F,Asian,Non-Hispanic,75024
4,1005,Robert,Johnson,1959-11-22,M,White,Non-Hispanic,75230


In [6]:
# Replace the selected columns if your schema uses different names.
summary_query = """
WITH patients_with_age AS (
    SELECT
        patient_id,
        EXTRACT(
            YEAR FROM AGE(CURRENT_DATE, date_of_birth)
        ) AS age
    FROM patients
    WHERE date_of_birth IS NOT NULL
)
SELECT
    COUNT(*) AS patient_count,
    ROUND(AVG(age), 1) AS mean_age,
    MIN(age) AS minimum_age,
    MAX(age) AS maximum_age
FROM patients_with_age;
"""

patient_summary = pd.read_sql(summary_query, engine)
patient_summary

,patient_count,mean_age,minimum_age,maximum_age
0,5,50.8,36.0,66.0


## Guided Laboratory 6 – SQL in Jupyter

Create and execute notebook queries that:

1. Preview each relevant table.
2. Retrieve a patient cohort using `WHERE`.
3. Sort the cohort using `ORDER BY`.
4. Calculate at least three aggregate measures.
5. Group results by a categorical variable.
6. Join patients to a clinical table.
7. Store each result in a clearly named DataFrame.
8. Display the first five rows and the DataFrame shape.

In [7]:
# Student workspace: write a cohort query.
cohort_query = """
SELECT *
FROM patients
LIMIT 20;
"""

cohort_df = pd.read_sql(cohort_query, engine)
print("Shape:", cohort_df.shape)
cohort_df.head()

Shape: (5, 8)


,patient_id,first_name,last_name,date_of_birth,sex,race,ethnicity,zip_code
0,1001,John,Doe,1968-04-12,M,White,Non-Hispanic,75201
1,1002,Maria,Lopez,1975-09-30,F,Hispanic,Hispanic,75204
2,1003,James,Smith,1982-01-18,M,Black,Non-Hispanic,75080
3,1004,Linda,Chen,1990-06-05,F,Asian,Non-Hispanic,75024
4,1005,Robert,Johnson,1959-11-22,M,White,Non-Hispanic,75230


In [8]:
# Student workspace: write a grouped summary query.
grouped_query = """
SELECT
    sex,
    COUNT(*) AS patient_count
FROM patients
GROUP BY sex
ORDER BY patient_count DESC;
"""

grouped_df = pd.read_sql(grouped_query, engine)
grouped_df

,sex,patient_count
0,M,3
1,F,2


# Practice Laboratory

Complete the following SQL tasks. Begin in the terminal, then repeat at least five in the notebook.

1. Display the first 10 patient records.
2. List all columns in the `patients` table.
3. Return distinct sex categories.
4. Count the total number of patients.
5. Find the youngest and oldest ages.
6. Calculate the average age.
7. Retrieve patients age 65 or older.
8. Retrieve patients between ages 45 and 64.
9. Find records with a missing value in a selected clinical variable.
10. Sort patients from oldest to youngest.
11. Return the five oldest patients.
12. Count patients by sex.
13. Calculate average age by sex.
14. Count patients by diabetes, smoking, or hypertension status if available.
15. Use `HAVING` to retain groups with at least 10 records.
16. Join `patients` and `bp_readings`.
17. Identify the highest systolic blood-pressure readings.
18. Join `patients` and `medications`.
19. Identify patients without a medication record using a `LEFT JOIN` and `IS NULL`.
20. Export one query result from pandas to `module3_query_result.csv`.

In [9]:
# Export example. Replace cohort_df with the DataFrame you want to submit.
cohort_df.to_csv("module3_query_result.csv", index=False)
print("Saved module3_query_result.csv")

Saved module3_query_result.csv


# Exploration Laboratory

Use the available PostgreSQL tables to answer **five clinically meaningful questions** of your choice.

For each question, provide:

1. The question in plain language.
2. The table or tables required.
3. The unit of analysis.
4. The SQL query.
5. The result.
6. A one- or two-sentence interpretation.
7. One limitation or assumption.

At least one question must use a join and at least one must use `GROUP BY`.

# Clinical Decision Point

A physician requests:

> **Give me all diabetic patients admitted last year.**

Before writing the query, identify:

- how diabetes is represented;
- which table defines patients;
- which table contains admissions or encounters;
- whether one patient may have multiple admissions;
- the exact start and end dates;
- whether observation and emergency encounters are included;
- whether the result should contain patients or admissions;
- how missing diagnosis information will be handled.

Write the SQL only after documenting these assumptions. Then explain how changing one assumption could change the answer.

# Reflection

In Module 2, you evaluated whether data could be trusted. In this module, you determined which data should be retrieved.

Respond briefly:

1. Which SQL concept was easiest to understand?
2. Which concept required the most careful reasoning?
3. How can a query be technically correct but clinically misleading?
4. What checks should be completed before sharing query results?

# Knowledge Check

1. What is the difference between a database and a table?
2. What is the role of a primary key?
3. What is the role of a foreign key?
4. What does `SELECT` do?
5. What does `WHERE` do?
6. Why is `IS NULL` used instead of `= NULL`?
7. What is the difference between `WHERE` and `HAVING`?
8. What does `GROUP BY` accomplish?
9. What is the difference between an `INNER JOIN` and a `LEFT JOIN`?
10. Why can a join increase the number of rows?
11. Why should analysts use `LIMIT` when exploring a large table?
12. What does one row represent after joining patients to repeated blood-pressure readings?
13. How does `pandas.read_sql()` support analysis?
14. Why should database credentials normally be stored securely?
15. What information should accompany a query used for a clinical cohort?

# Connection to the Group Project

For this milestone, your team will:

1. Identify the PostgreSQL table or tables relevant to the project.
2. Document primary and foreign keys.
3. Define the unit of analysis.
4. Write at least five SQL queries that answer meaningful project questions.
5. Include at least one filtered cohort, one grouped summary, and one join.
6. Save the final analytical cohort as a pandas DataFrame.
7. Export the cohort for use in Module 4.
8. Document all inclusion criteria, exclusions, assumptions, and known limitations.

# Module Summary

In this module, you moved from spreadsheet-based exploration to relational data retrieval. You connected to PostgreSQL through the terminal, inspected database structures, wrote foundational SQL queries, filtered and summarized records, combined related tables, and brought query results into a Jupyter Notebook as pandas DataFrames.

The central principle is:

> **SQL should translate a clearly defined healthcare question into a transparent and reproducible data request.**

**Next module:** Cleaning and Exploring Healthcare Data with Python and pandas.